# Test Movie Idea

## Load dependencies

In [1]:
import kagglehub
from pathlib import Path
import pandas as pd
import numpy as np
import ast

from params import SEP, ENCODING

/work/NLP_Exam_Idea/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [2]:
# Download latest version of the Cornell Movie Dataset
path = kagglehub.dataset_download("rajathmc/cornell-moviedialog-corpus")

print("Path to dataset files:", path)

100%|██████████| 9.58M/9.58M [00:00<00:00, 12.9MB/s]

Extracting files...


Path to dataset files: /home/ucloud/.cache/kagglehub/datasets/rajathmc/cornell-moviedialog-corpus/versions/1


In [3]:
# define the path to the dataset folder using the Path module
dataset_dir = Path(path)

# print all the files within the dataset folder
for file in dataset_dir.iterdir():
    print(file.name)

.DS_Store
README.txt
chameleons.pdf
movie_characters_metadata.txt
movie_conversations.txt
movie_lines.txt
movie_titles_metadata.txt
raw_script_urls.txt


In [4]:

# loading the three datasets with respectively dialogue lines, character metadata and movie metadata

def load_file(dataset_dir, filename, columns):
    """
    Load a dataset file using the delimiter and encoding specified in the data documentation.
    """
    return pd.read_csv(
        Path(dataset_dir) / filename,
        sep=SEP,
        engine="python",
        header=None,
        names=columns,
        encoding=ENCODING,
    )


def load_datasets(dataset_dir):
    """
    Load the datasets with the raw dialogue text, character metadata, and movie metadata.
    """
    df = load_file(dataset_dir,
        filename = "movie_lines.txt",
        columns= ["line_id", "character_id", "movie_id", "character_name", "text"],
    )

    characters = load_file(dataset_dir,
        filename = "movie_characters_metadata.txt",
        columns = ["character_id", "character_name", "movie_id","movie_title", "gender", "character_position", ],
    )

    movies = load_file(dataset_dir,
        filename = "movie_titles_metadata.txt",
        columns = ["movie_id", "movie_title", "movie_year", "rating", "votes", "genres"],
    )

    conversations = load_file(dataset_dir, 
    filename = "movie_conversations.txt",
    columns = ["char1", "char2", "movie_id", "utterance_ids"],
    )

    return df, characters, movies, conversations


def merge_metadata(df, characters, movies):
    """
    Add character gender and movie year to dialogue lines.
    """
    character_gender = characters[["character_id", "gender"]].drop_duplicates("character_id")
    movie_year = movies[["movie_id", "movie_year"]].drop_duplicates("movie_id")

    df = df.merge(
        character_gender,
        on="character_id",
        how="left",
        validate="many_to_one",
    )

    df = df.merge(
        movie_year,
        on="movie_id",
        how="left",
        validate="many_to_one",
    )

    return df

In [5]:
df, characters, movies, conversations = load_datasets(dataset_dir)
df = merge_metadata(df, characters, movies)

In [6]:
df.head()

,line_id,character_id,movie_id,character_name,text,gender,movie_year
0,L1045,u0,m0,BIANCA,They do not!,f,1999
1,L1044,u2,m0,CAMERON,They do to!,m,1999
2,L985,u0,m0,BIANCA,I hope so.,f,1999
3,L984,u2,m0,CAMERON,She okay?,m,1999
4,L925,u0,m0,BIANCA,Let's go.,f,1999


## Preprocessing

In [7]:
def normalize(df):
    """ Normalize and clean"""
    df = df.copy()

    # normalize IDs and gender
    id_cols = ["line_id", "character_id", "movie_id"]
    df[id_cols] = df[id_cols].astype("string").apply(lambda col: col.str.strip())

    df["gender"] = df["gender"].astype("string").str.strip().str.upper()

    # clean text
    df["text"] = df["text"].astype("string").str.strip()

    # convert movie year to numeric
    df["movie_year"] = pd.to_numeric(df["movie_year"], errors="coerce")
    return df

In [8]:


# make a function that can take the data from the conversations txt and infer the adressee of the utterance

def build_target_df(df, conversations):
    """
    Use the conversation 
    Make a dataframe with one row per utterance and the following columns:
      text              : the utterance
      speaking_gender   : gender of current speaker
      target_gender     : inferred gender of addressee
      speaker_id        : current speaker's character ID
      target_id         : inferred addressee's character ID
      movie_id          : movie ID
      movie_year        : movie release year
      line_id           : utterance ID
      conversation_id   : conversation index
      turn_index        : chronological position in the conversation
    """

    # make lookups for dialogue lines and unambiguous character genders
    lines = df.drop_duplicates("line_id").set_index("line_id")
    genders = df.groupby("character_id")["gender"].agg(
        lambda s: s.dropna().unique()[0] if s.dropna().nunique() == 1 else pd.NA
    ).to_dict()

    rows = []

    # go through each conversation and its utterances in order
    for conv_id, conv in conversations.iterrows():
        try:
            line_ids = ast.literal_eval(conv["utterance_ids"])
        except (ValueError, SyntaxError, TypeError):
            continue

        # get the participants and movie ID
        char1, char2, movie_id = (
            str(conv[c]).strip() for c in ["char1", "char2", "movie_id"]
        )

        for turn, line_id in enumerate(line_ids):
            line_id = str(line_id).strip()

            # skip missing lines and lines belonging to another movie
            if line_id not in lines.index:
                continue

            line = lines.loc[line_id]
            if line["movie_id"] != movie_id:
                continue

            # infer the addressee as the other conversation participant
            speaker = line["character_id"]
            target = char2 if speaker == char1 else char1 if speaker == char2 else pd.NA

            # save the utterance, speaker, addressee, and metadata
            rows.append({
                "conversation_id": conv_id,
                "turn_index": turn,
                "movie_id": movie_id,
                "movie_year": line["movie_year"],
                "line_id": line_id,
                "speaker_id": speaker,
                "target_id": target,
                "text": line["text"],
                "speaking_gender": genders.get(speaker, pd.NA),
                "target_gender": genders.get(target, pd.NA),
            })

    # one row per utterance
    return pd.DataFrame(rows)

In [9]:

# Build the utterance-level dataset with inferred addressees
df_normalized = normalize(df)
addressed_df = build_target_df(df_normalized, conversations)


# Inspect the output
print(addressed_df.shape)
print(addressed_df["target_gender"].value_counts(dropna=False))
print(addressed_df["speaking_gender"].value_counts(dropna=False))

(304713, 10)
target_gender
M    170878
F     71589
?     62246
Name: count, dtype: int64
speaking_gender
M    170768
F     71255
?     62690
Name: count, dtype: int64


In [10]:
addressed_df.head()

,conversation_id,turn_index,movie_id,movie_year,line_id,speaker_id,target_id,text,speaking_gender,target_gender
0,0,0,m0,1999.0,L194,u0,u2,Can we make this quick? Roxanne Korrine and A...,F,M
1,0,1,m0,1999.0,L195,u2,u0,"Well, I thought we'd start with pronunciation,...",M,F
2,0,2,m0,1999.0,L196,u0,u2,Not the hacking and gagging and spitting part....,F,M
3,0,3,m0,1999.0,L197,u2,u0,Okay... then how 'bout we try out some French ...,M,F
4,1,0,m0,1999.0,L198,u0,u2,You're asking me out. That's so cute. What's ...,F,M


## Make descriptive stats

In [11]:
def descriptive_stats(df):
    """Summarize the target-gender dialogue dataset."""
    stats = pd.DataFrame({
        "Statistic": [
            "Number of utterances",
            "Number of conversations",
            "Number of movies",
            "Number of speakers",
            "Number of addressees",
            "Female addressees (%)",
            "Male addressees (%)",
            "Mean utterance length (words)",
            "Median utterance length (words)",
            "Mean movie year",
            "Earliest movie year",
            "Latest movie year",
        ],
        "Value": [
            len(df),
            df["conversation_id"].nunique(),
            df["movie_id"].nunique(),
            df["speaker_id"].nunique(),
            df["target_id"].nunique(),
            round(df["target_gender"].eq("F").mean() * 100, 2),
            round(df["target_gender"].eq("M").mean() * 100, 2),
            round(df["text"].str.split().str.len().mean(), 2),
            round(df["text"].str.split().str.len().median(), 2),
            int(round(df["movie_year"].mean())),
            int(df["movie_year"].min()),
            int(df["movie_year"].max()),
        ],
    })

    return stats

In [16]:
stats = descriptive_stats(addressed_df)

# save
stats.to_csv("../plots/descriptive_stats.csv")


# show a prettier table
display(stats.style.format({"Value": "{:g}"}))

,Statistic,Value
0,Number of utterances,304713
1,Number of conversations,83097
2,Number of movies,617
3,Number of speakers,9035
4,Number of addressees,9035
5,Female addressees (%),23.49
6,Male addressees (%),56.08
7,Mean utterance length (words),10.55
8,Median utterance length (words),7
9,Mean movie year,1988
